In [1]:
import os
import time
import json
import pandas as pd
from langchain_google_genai import ChatGoogleGenerativeAI
from langchain_core.prompts import ChatPromptTemplate
from pydantic import BaseModel, Field
from typing import List
from dotenv import load_dotenv

c:\Users\pawan\Desktop\AI_KBG\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
# Load environment variables
load_dotenv()

True

In [3]:
# Define Pydantic models for Extraction for Structured data
class Entity(BaseModel):
    message_id: str = Field(description="Email ID")
    entity: str = Field(description="Exact text of the entity")
    entity_type: str = Field(description="Uppercase category label (e.g., PERSON, ORGANIZATION, COMMODITY, REGULATION)")

class Relationship(BaseModel):
    message_id: str = Field(description="Email ID")
    subject: str = Field(description="Source entity name (Must match an extracted entity)")
    predicate: str = Field(description="Uppercase verb phrase with underscores (e.g., WORKS_FOR, TRADED_BY)")
    object: str = Field(description="Target entity name (Must match an extracted entity)")

class BatchExtraction(BaseModel):
    entities: List[Entity]
    relationships: List[Relationship]

In [4]:
# Initialize LangChain Gemini model
api_key = os.getenv("GEMINI_AI_API_KEY")
if not api_key:
    raise ValueError("GEMINI_AI_API_KEY not found in environment variables.")

In [ ]:
# Set up prompt
prompt = ChatPromptTemplate.from_template(
    """
    Extract exhaustive Knowledge Graph triples (Entities and Relationships) from the provided batch of emails.
    
    CRITICAL BATCH INSTRUCTION: 
    The `Batch Data` contains MULTIPLE independent emails bounded by `--- EMAIL START ---` and `--- EMAIL END ---`.
    Each email has a unique `ID`. You MUST process each strictly independently.

    ### EXTRACTION RULES:
    1. ISOLATION: Never link entities across different emails.
    2. CONGRUENCE: Every `subject` and `object` in a relationship MUST mathematically exist in the `entities` list.
    3. ACCURACY: Use the exact email `ID` for every extracted item. No hallucinations.
    4. NOISE: Ignore system logs, automated messages, and generic boilerplates (e.g., "eserver@").
    5. UNBIASED EXTRACTION: The category targets below are strictly examples. You MUST NOT restrict yourself to these lists. Freely extract ANY relevant entity or relationship.

    ### EXTRACTION TARGETS:
    **Entities (Focus on high-value extractions):**
    - News & Media: PERSON, ORGANIZATION, GEOGRAPHIC_LOCATION, STRATEGY, EVENT
    - Energy Trading: COMMODITY, PLANT, POWER_GRID, ISO_RTO, HUB
    - Legal & Regulatory: CASE_NUMBER, LAW_FIRM, JURISDICTION, REGULATION, LEGAL_TERM
    - HR & Internal: JOB_TITLE, RESUME_ID, BENEFIT_PLAN, OFFER_STATUS
    - Financial & Accounting: ACCOUNT_NUMBER, CURRENCY, TRANSACTION_TYPE, ASSET_CLASS, DEAL_ID
    - Dynamic: Extract any other relevant entity with a concise, UPPERCASE label.

    **Relationships (Format: Subject -[PREDICATE]-> Object):**
    - News & Media: WORKS_FOR, LOCATED_IN, PARTICIPANT_IN, HAS_COMMUNICATION, PUBLISHED_BY
    - Energy Trading: TRADED_BY, PRICE_FOR, HEDGED_WITH, SUPPLIED_TO
    - Legal & Regulatory: SUBPOENAED_BY, COMPLIES_WITH, LITIGATED_BY, REPRESENTED_BY
    - HR & Internal: REPORTS_TO, APPROVED_BY, HIRED_FOR, STAKEHOLDER_IN
    - Financial & Accounting: INVOICED_BY, AUDITED_IN, BALANCE_OF, TRANSFERRED_TO
    - Dynamic: Generate precise UPPERCASE verb phrases using UNDERSCORES (e.g., REPORTED_TO). DO NOT restrict yourself to predefined lists.

    >> BATCH PROCESSING: Treat each `--- EMAIL START ---` block as an isolated universe. Use its `ID:` as the `message_id`.

    Batch Data:
    ---
    {text_batch}
    ---
    """
)

In [6]:
# Initialize LangChain Gemini model (Global Instance)
llm = ChatGoogleGenerativeAI(
    model="gemini-3.1-flash-lite-preview",
    google_api_key=api_key,
    max_output_tokens=8192
)

In [7]:
# Using with_structure_output to follow the pydantic structure by llm
chain = prompt | llm.with_structured_output(BatchExtraction)

In [8]:
def extract_batch(batch_rows):
    """
    Extracts entities and relationships from a batch of emails.
    """

    def safe_get(row, col):
        val = row.get(col, '')
        return str(val).strip() if val and str(val).lower() not in ('nan', 'none', '') else 'Unknown'

    text_batch = "\n\n".join([
        f"--- EMAIL START ---\n"
        f"ID: {safe_get(row, 'message_id')}\n"
        f"Subject: {safe_get(row, 'subject')}\n"
        f"Body: {safe_get(row, 'body_cleaned')}\n"
        f"--- EMAIL END ---"
        for _, row in batch_rows.iterrows()
    ])
    
    attempt = 0
    base_delay = 10
    while True:
        try:
            print(f"Submitting batch of {len(batch_rows)} emails to LLM for NER...")
            
            response = chain.invoke({
                "text_batch": text_batch
            })
            
            # response is a BatchExtraction instance or dict depending on LangChain version
            if hasattr(response, "entities"):
                entities = [e.model_dump() for e in response.entities]
                relationships = [r.model_dump() for r in response.relationships]
            else:
                entities = response.get("entities", [])
                relationships = response.get("relationships", [])
            print(f"Received {len(entities)} entities and {len(relationships)} relationships.")
            return entities, relationships
            
        except Exception as e:
            error_str = str(e)
            if "429" in error_str or "RESOURCE_EXHAUSTED" in error_str:
                attempt += 1
                delay = min(base_delay * (2 ** (attempt - 1)), 300)
                print(f"Rate limit hit. Retrying in {delay}s... (Total Retries: {attempt})")
                time.sleep(delay)
            elif "output parsing" in error_str.lower() or "json" in error_str.lower():
                # On parsing error, return None to trigger a retry with higher temperature in the caller
                print(f"JSON Parsing Error: {e}")
                return None, None
            else:
                print(f"Error extracting batch: {e}")
                return None, None

In [9]:
def load_checkpoint(checkpoint_file):
    if not os.path.exists(checkpoint_file):
        save_checkpoint(checkpoint_file, -1)
        
    try:
        with open(checkpoint_file, 'r') as f:
            content = f.read().strip()
            if not content:
                return -1
            return json.loads(content).get("last_processed_index", -1)
    except Exception:
        return -1

In [10]:
def save_checkpoint(checkpoint_file, last_index):
    with open(checkpoint_file, 'w') as f:
        json.dump({"last_processed_index": last_index, "timestamp": time.ctime()}, f)

In [11]:
def process_emails_serial(input_csv, entities_csv, relationships_csv, checkpoint_file, batch_size=10):
    """
    Process emails in batches for NER and Relationship extraction.
    """
    # Ensure directories exist
    os.makedirs(os.path.dirname(entities_csv), exist_ok=True)
    os.makedirs(os.path.dirname(relationships_csv), exist_ok=True)
    print(f"Loading {input_csv}...")
    df = pd.read_csv(input_csv)
    df = df[df['body_cleaned'].notna()].reset_index(drop=True)  # serial index
    
    total_records = len(df)
    last_index = load_checkpoint(checkpoint_file)

    # Auto-reset: if output files were deleted, restart from scratch
    if not os.path.exists(entities_csv) or not os.path.exists(relationships_csv):
        if last_index >= 0:
            print("WARNING: Output CSV file(s) missing but checkpoint exists. Resetting to start from index 0.")
            if os.path.exists(checkpoint_file):
                os.remove(checkpoint_file)
        last_index = -1

    start_index = last_index + 1
    print(f"Resuming NER extraction from index {start_index} out of {total_records}...")


    # Rate Limits: Strictly 15 RPM max (4 seconds per request)
    min_request_interval = 60.0 / 15.0 
    last_request_start = 0

    if start_index >= total_records:
        print("All emails processed.")
        return

    i = start_index
    batch_retries = 0
    while i < total_records:
        elapsed = time.time() - last_request_start
        if elapsed < min_request_interval:
            sleep_time = min_request_interval - elapsed
            print(f"Rate Limiting: Pausing for {sleep_time:.2f}s to maintain strictly 15 RPM...")
            time.sleep(sleep_time)
        
        last_request_start = time.time()
        
        end_idx = min(i + batch_size, total_records)
        batch_rows = df.iloc[i : end_idx]


        print(f"\n--- Processing BATCH: Indices {i} to {end_idx-1} ---")
        entities, relationships = extract_batch(batch_rows)
        
        # Build a serial order map: message_id -> position in this batch
        batch_order = {row['message_id']: idx for idx, (_, row) in enumerate(batch_rows.iterrows())}
        
        if entities is not None and relationships is not None:
            if entities:
                ent_triples = []
                for ent in entities:
                    ent_data = ent if isinstance(ent, dict) else ent.model_dump()
                    msg_id = ent_data.get('message_id', 'Unknown')
                    ent_triples.append({
                        "predicate": "HAS_ENTITY",
                        "object": ent_data.get('entity', 'Unknown'),
                        "entity_type": ent_data.get('entity_type', 'Entity'),
                        "message_id": msg_id,
                        "_sort_key": batch_order.get(msg_id, 9999)  # for ordering
                    })
                if ent_triples:
                    ent_df = pd.DataFrame(ent_triples)
                    ent_df = ent_df.sort_values('_sort_key').drop(columns=['_sort_key'])
                    # Ensure column order: predicate, object, entity_type, message_id
                    ent_df = ent_df[['predicate', 'object', 'entity_type', 'message_id']]
                    ent_df.to_csv(entities_csv, mode='a', index=False, header=not os.path.exists(entities_csv))
            
            if relationships:
                rel_triples = []
                for rel in relationships:
                    rel_data = rel if isinstance(rel, dict) else rel.model_dump()
                    msg_id = rel_data.get('message_id', 'Unknown')
                    rel_triples.append({
                        "message_id": msg_id,
                        "subject": rel_data.get('subject', 'Unknown'),
                        "predicate": rel_data.get('predicate', 'RELATED_TO'),
                        "object": rel_data.get('object', 'Unknown'),
                        "_sort_key": batch_order.get(msg_id, 9999)  # for ordering
                    })
                if rel_triples:
                    rel_df = pd.DataFrame(rel_triples)
                    rel_df = rel_df.sort_values('_sort_key').drop(columns=['_sort_key'])
                    # Ensure column order: message_id, subject, predicate, object
                    rel_df = rel_df[['message_id', 'subject', 'predicate', 'object']]
                    rel_df.to_csv(relationships_csv, mode='a', index=False, header=not os.path.exists(relationships_csv))
                
            save_checkpoint(checkpoint_file, end_idx - 1)
            print(f"Processed email index {end_idx-1}. Checkpoint updated.")
            
            # ADVANCE TO THE NEXT BATCH
            i = end_idx
            batch_retries = 0
            
        else:
            batch_retries += 1
            print(f"Failed batch at index {i}. Retrying SAME BATCH in 10 seconds to ensure no data is skipped... (Retry {batch_retries})")
            time.sleep(10)
            
    print(f"\nExtraction complete. Entities: {entities_csv}, Relationships: {relationships_csv}")


In [12]:
if __name__ == "__main__":
    # Settings for the sampling task
    input_path = "sample_email_by_category/sample_email.csv"
    entities_path = "NER/entities/entities.csv"
    relationships_path = "NER/relationships/relationships.csv"
    checkpoint_path = "extraction_checkpoint.json"
    
    # Batch sizing=10 for better result
    process_emails_serial(input_path, entities_path, relationships_path, checkpoint_path, batch_size=10)

Loading sample_email_by_category/sample_email.csv...
Resuming NER extraction from index 1000 out of 1000...
All emails processed.
